In [34]:
import yaml
import glob
import json
import os
from dataclasses import dataclass
from pprint import pprint
from typing import Dict, Optional, List, Any
from slideguard.schemes import FullEvaluation
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.language_models import LanguageModelInput


/home/nikolay/.cache/pypoetry/virtualenvs/slideguard-UU6zwuLG-py3.12/lib/python3.12/site-packages/pydantic/_internal/_config.py:373: UserWarning: Valid config keys have changed in V2:
* 'allow_population_by_field_name' has been renamed to 'validate_by_name'
  warnings.warn(message, UserWarning)


PydanticUserError: The `__modify_schema__` method is not supported in Pydantic v2. Use `__get_pydantic_json_schema__` instead in class `SecretStr`.

For further information visit https://errors.pydantic.dev/2.11/u/custom-json-schema

# Functions

In [35]:
class VLLMChatOpenAI(ChatOpenAI):
    def _get_request_payload(
        self,
        input_: LanguageModelInput,
        *,
        stop: Optional[List[str]] = None,
        **kwargs: Any,
    ) -> dict:
        payload = super()._get_request_payload(input_, stop=stop, **kwargs)
        # max_tokens was deprecated in favor of max_completion_tokens
        # in September 2024 release
        if "max_completion_tokens" in payload:
            payload["max_tokens"] = payload.pop("max_completion_tokens")
        return payload
    

def load_goldens(base_path: str = "../golden") -> Dict[str, dict]:
    golden_files = glob.glob(os.path.join(base_path, "*.yaml"))

    goldens = dict()
    for file in golden_files:
        deck_name, _ = os.path.splitext(os.path.basename(file))
        with open(file, "r") as f:
            evaluation = yaml.safe_load(f)
        goldens[deck_name] = evaluation
    
    return goldens


def load_evaluations(base_path: str = "../slidedecks_test_evaluations") -> Dict[str, FullEvaluation]:
    evaluation_files = glob.glob(os.path.join(base_path, "evaluations_*.json"))

    evaluations = dict()
    for file in evaluation_files:
        deck_name, _ = os.path.splitext(os.path.basename(file))
        deck_name = deck_name.replace("evaluations_", "")
        with open(file, "r") as f:
            evaluation = FullEvaluation.model_validate_json(f.read())
        evaluations[deck_name] = evaluation
    
    return evaluations

NameError: name 'ChatOpenAI' is not defined

In [36]:
goldens = load_goldens()
evaluations = load_evaluations()

print("Goldens:", len(goldens))
print("Evaluations:", len(evaluations))

Goldens: 3
Evaluations: 3


In [37]:
golden2evaluation = dict()
for deck_name, golden in goldens.items():
    if deck_name in evaluations:
        golden2evaluation[deck_name] = (golden, evaluations[deck_name])
    else:
        print(f"No evaluation for {deck_name}")

print(len(golden2evaluation))

3


In [38]:
golden2evaluation.keys()

dict_keys(['1_EN_Kataeva_Thesis', '15_RU_Basilaev_Thesis', '12_EN_Zamiralov_NIR'])

In [50]:
gold, evaluation = golden2evaluation['1_EN_Kataeva_Thesis']

ev_results = dict()

def _convert(el):
    del el['severity']
    return el

for criteria, evaluations in evaluation.deck_evaluations.evaluations.items():
    eval_results = [_convert(el) for el in evaluations['evaluation_results'] if el['severity'] > 2]
    ev_results[criteria.value] = eval_results

pprint(ev_results)

{'deck_research_quality': [{'evaluation_element': 'Scientific rigor of the '
                                                  'research',
                            'evaluation_suggestion': 'The research employs '
                                                     'appropriate scientific '
                                                     'methods and metrics. '
                                                     'However, the '
                                                     'presentation could '
                                                     'elaborate more on the '
                                                     'statistical significance '
                                                     'of the results and '
                                                     'discuss potential '
                                                     'limitations and future '
                                                     'work to further solidify '
                

['Отсутствует обзор литературы (альтернативные подходы)',
 'Не хватает слайда с обзором конкурирующих решений']